# 🏦 Loan Approval Prediction (Classification)
**Target:** `Loan_Status` — Y / N

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 2. Load Dataset

In [ ]:
try:
    df = pd.read_csv('loan_data.csv')
except FileNotFoundError:
    print("CSV not found."); raise
print(f"Shape: {df.shape}"); df.head()

## 3. Identify Data Types

In [ ]:
print("Data types:"); print(df.dtypes)
print(f"\nNumeric : {df.select_dtypes(include='number').columns.tolist()}")
print(f"Object  : {df.select_dtypes(include='object').columns.tolist()}")

## 4. Descriptive Statistics

In [ ]:
df.describe().round(2)

## 5. Handle Missing Values

In [ ]:
print("Missing:"); print(df.isnull().sum()[df.isnull().sum()>0])
for col in ['Gender','Married','Dependents','Self_Employed','Loan_Amount_Term','Credit_History']:
    if df[col].isnull().sum()>0:
        m=df[col].mode()[0]; df[col].fillna(m,inplace=True); print(f"  '{col}' → mode: {m}")
if df['LoanAmount'].isnull().sum()>0:
    m=df['LoanAmount'].median(); df['LoanAmount'].fillna(m,inplace=True); print(f"  'LoanAmount' → median: {m}")
print(f"Remaining: {df.isnull().sum().sum()}")

## 6. Handle Duplicates

In [ ]:
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True); print(f"Shape: {df.shape}")

## 7. Outlier Detection & Handling

In [ ]:
num_cols=['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term']
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,col in zip(axes,num_cols):
    ax.boxplot(df[col],vert=False,patch_artist=True,boxprops=dict(facecolor='steelblue',alpha=0.6))
    ax.set_title(col,fontsize=9); ax.set_yticks([])
plt.tight_layout(); plt.show()
before=len(df)
for col in ['ApplicantIncome','CoapplicantIncome','LoanAmount']:
    Q1,Q3=df[col].quantile([0.25,0.75]); IQR=Q3-Q1
    df=df[df[col].between(Q1-1.5*IQR,Q3+1.5*IQR)]
print(f"Removed: {before-len(df)} | Shape: {df.shape}")

## 8. Visualizations & Insights

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
counts=df['Loan_Status'].value_counts()
axes[0].bar(counts.index,counts.values,color=['#2ecc71','#e74c3c'],edgecolor='white',width=0.4)
axes[0].set_title('Loan Status Distribution')
for i,(l,v) in enumerate(counts.items()): axes[0].text(i,v+3,str(v),ha='center',fontweight='bold')

ct=pd.crosstab(df['Credit_History'],df['Loan_Status'],normalize='index')*100
ct.plot(kind='bar',ax=axes[1],color=['#e74c3c','#2ecc71'],edgecolor='white',rot=0)
axes[1].set_title('Approval by Credit History'); axes[1].set_xlabel('Credit History'); axes[1].legend(['Rejected','Approved'],fontsize=8)

df.boxplot(column='LoanAmount',by='Loan_Status',ax=axes[2])
plt.sca(axes[2]); plt.title('Loan Amount by Status')
plt.suptitle(''); plt.tight_layout(); plt.show()
print("""Insights:
1. ~70% approved vs ~30% rejected — SMOTE applied to balance.
2. Good credit history (1) leads to ~80% approval rate.
3. Rejected applicants tend to request higher loan amounts.""")

## 9. Feature Engineering, Encode & Scale

In [ ]:
df.drop(columns=['Loan_ID'],inplace=True)
df['Total_Income']=df['ApplicantIncome']+df['CoapplicantIncome']
df['Income_to_Loan']=df['Total_Income']/(df['LoanAmount']+1)
df['LoanAmount_log']=np.log1p(df['LoanAmount'])
df['Total_Income_log']=np.log1p(df['Total_Income'])

le=LabelEncoder(); df['Loan_Status']=le.fit_transform(df['Loan_Status'])
cat_cols=df.select_dtypes(include='object').columns.tolist()
print(f"Encoding: {cat_cols}")

# Save label encoders for prediction
label_encoders={}
for col in cat_cols:
    lenc=LabelEncoder(); df[col]=lenc.fit_transform(df[col].astype(str)); label_encoders[col]=lenc

drop_cols=['ApplicantIncome','CoapplicantIncome','LoanAmount','Total_Income']
X=df.drop(columns=['Loan_Status']+drop_cols); y=df['Loan_Status']
FEATURE_COLS=X.columns.tolist()

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
smote=SMOTE(random_state=42); X_train_res,y_train_res=smote.fit_resample(X_train,y_train)
print(f"After SMOTE: {dict(pd.Series(y_train_res).value_counts())}")
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train_res); X_test_sc=scaler.transform(X_test)
print(f"Train: {X_train_sc.shape[0]} | Test: {X_test_sc.shape[0]}")

## 10. Model Building

In [ ]:
models={
    'Logistic Regression': LogisticRegression(max_iter=1000,random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5,random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100,max_depth=6,random_state=42)
}
results={}; preds={}; probs={}
for name,model in models.items():
    model.fit(X_train_sc,y_train_res)
    yp=model.predict(X_test_sc); ypr=model.predict_proba(X_test_sc)[:,1]
    preds[name]=yp; probs[name]=ypr
    results[name]={'Accuracy':round(accuracy_score(y_test,yp),4),'ROC-AUC':round(roc_auc_score(y_test,ypr),4)}
    print(f"\n{'='*40}\n  {name}\n{'='*40}")
    print(classification_report(y_test,yp,target_names=['Rejected','Approved']))

## 11. Model Comparison

In [ ]:
res_df=pd.DataFrame(results).T.sort_values('ROC-AUC',ascending=False); print(res_df)
colors=['#3498db','#e67e22','#2ecc71']
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].bar(res_df.index,res_df['Accuracy'],color=colors,edgecolor='white',width=0.4)
axes[0].set_title('Accuracy',fontweight='bold'); axes[0].set_ylim(0,1)
axes[0].set_xticklabels(res_df.index,rotation=15,ha='right',fontsize=9)
axes[1].bar(res_df.index,res_df['ROC-AUC'],color=colors,edgecolor='white',width=0.4)
axes[1].set_title('ROC-AUC',fontweight='bold'); axes[1].set_ylim(0,1)
axes[1].set_xticklabels(res_df.index,rotation=15,ha='right',fontsize=9)
for (name,ypr),color in zip(probs.items(),colors):
    fpr,tpr,_=roc_curve(y_test,ypr)
    axes[2].plot(fpr,tpr,label=f"{name} ({results[name]['ROC-AUC']:.3f})",color=color,linewidth=2)
axes[2].plot([0,1],[0,1],'k--',linewidth=1); axes[2].set_title('ROC Curves')
axes[2].legend(fontsize=8)
plt.suptitle('Model Comparison',fontsize=13,y=1.02); plt.tight_layout(); plt.show()

---
## 🔮 12. Predict Loan Approval for Your Own Applicant
**Edit the values below and run the cell.**

In [ ]:
# ╔══════════════════════════════════════════╗
# ║   ✏️  CHANGE THESE VALUES TO YOUR INPUT  ║
# ╚══════════════════════════════════════════╝

gender           = 'Male'         # 'Male' or 'Female'
married          = 'Yes'          # 'Yes' or 'No'
dependents       = '0'            # '0', '1', '2', '3+'
education        = 'Graduate'     # 'Graduate' or 'Not Graduate'
self_employed    = 'No'           # 'Yes' or 'No'
applicant_income  = 5000          # Monthly income ($)
coapplicant_income = 2000         # Co-applicant monthly income ($)
loan_amount      = 150            # Loan amount in $1000s
loan_term        = 360            # Loan term in months (e.g. 360=30yrs)
credit_history   = 1.0            # 1.0 = Good, 0.0 = Bad
property_area    = 'Urban'        # 'Urban', 'Semiurban', or 'Rural'

# ── Auto-process ─────────────────────────
total_income     = applicant_income + coapplicant_income
income_to_loan   = total_income / (loan_amount + 1)
loan_amount_log  = np.log1p(loan_amount)
total_income_log = np.log1p(total_income)

raw = {
    'Gender':gender, 'Married':married, 'Dependents':dependents,
    'Education':education, 'Self_Employed':self_employed,
    'Loan_Amount_Term':loan_term, 'Credit_History':credit_history,
    'Property_Area':property_area,
    'Income_to_Loan':income_to_loan,
    'LoanAmount_log':loan_amount_log,
    'Total_Income_log':total_income_log
}

new_row = pd.DataFrame([raw])
for col, lenc in label_encoders.items():
    if col in new_row.columns:
        try:
            new_row[col] = lenc.transform(new_row[col].astype(str))
        except ValueError:
            new_row[col] = 0

new_row = new_row.reindex(columns=FEATURE_COLS, fill_value=0)
new_scaled = scaler.transform(new_row)

print("=" * 48)
print("       🏦 LOAN APPROVAL PREDICTION RESULTS")
print("=" * 48)
print(f"  Applicant Income   : ${applicant_income:,}")
print(f"  Co-applicant Income: ${coapplicant_income:,}")
print(f"  Loan Amount        : ${loan_amount}k")
print(f"  Credit History     : {'Good ✓' if credit_history==1 else 'Bad ✗'}")
print(f"  Property Area      : {property_area}")
print("-" * 48)
for name, model in models.items():
    pred = model.predict(new_scaled)[0]
    prob = model.predict_proba(new_scaled)[0][1]
    label = '✅ APPROVED' if pred == 1 else '❌ REJECTED'
    print(f"  {name:<22}: {label}  (confidence: {prob:.2%})")
print("=" * 48)